In [5]:
import os
DATA_DIR = "/kaggle/input/datasets/nhttrangminh/deepfashion"
print(os.listdir(DATA_DIR))

['merged_dataset_clean.csv', 'img']


In [6]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nhttrangminh/deepfashion/merged_dataset_clean.csv")
df.head()

,image_path,category,split
0,img/Sheer_Pleated-Front_Blouse/img_00000001.jpg,3,train
1,img/Sheer_Pleated-Front_Blouse/img_00000002.jpg,3,train
2,img/Sheer_Pleated-Front_Blouse/img_00000003.jpg,3,val
3,img/Sheer_Pleated-Front_Blouse/img_00000004.jpg,3,train
4,img/Sheer_Pleated-Front_Blouse/img_00000005.jpg,3,test


In [7]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-o4hanh2e
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-o4hanh2e
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import clip
from PIL import Image
import pandas as pd
from tqdm import tqdm
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess = clip.load("ViT-B/32", device=device)

df = pd.read_csv("/kaggle/input/datasets/nhttrangminh/deepfashion/merged_dataset_clean.csv")

IMG_ROOT = "/kaggle/input/datasets/nhttrangminh/deepfashion/img"

image_embeddings = []
text_embeddings = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        img_path = os.path.join(IMG_ROOT, row["image_path"])
        image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

        text = f"a photo of a category {row['category']}"
        text_token = clip.tokenize([text]).to(device)

        with torch.no_grad():
            img_feat = model.encode_image(image)
            txt_feat = model.encode_text(text_token)

        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

        image_embeddings.append(img_feat.cpu())
        text_embeddings.append(txt_feat.cpu())

    except Exception as e:
        print(f"Error at {row['image_path']}: {e}")

image_embeddings = torch.cat(image_embeddings, dim=0)
text_embeddings = torch.cat(text_embeddings, dim=0)

print("Encode done:", image_embeddings.shape, text_embeddings.shape)

 85%|████████▌ | 246497/289222 [2:03:26<03:05, 230.66it/s]  

Error at img/Striped_A-line_Dress/img_00000001.jpg: [Errno 2] No such file or directory: '/kaggle/input/datasets/nhttrangminh/deepfashion/img/img/Striped_A-line_Dress/img_00000001.jpg'
Error at img/Striped_A-line_Dress/img_00000002.jpg: [Errno 2] No such file or directory: '/kaggle/input/datasets/nhttrangminh/deepfashion/img/img/Striped_A-line_Dress/img_00000002.jpg'
Error at img/Striped_A-line_Dress/img_00000003.jpg: [Errno 2] No such file or directory: '/kaggle/input/datasets/nhttrangminh/deepfashion/img/img/Striped_A-line_Dress/img_00000003.jpg'
Error at img/Striped_A-line_Dress/img_00000004.jpg: [Errno 2] No such file or directory: '/kaggle/input/datasets/nhttrangminh/deepfashion/img/img/Striped_A-line_Dress/img_00000004.jpg'
Error at img/Striped_A-line_Dress/img_00000005.jpg: [Errno 2] No such file or directory: '/kaggle/input/datasets/nhttrangminh/deepfashion/img/img/Striped_A-line_Dress/img_00000005.jpg'
Error at img/Striped_A-line_Dress/img_00000006.jpg: [Errno 2] No such file 

100%|██████████| 289222/289222 [2:25:20<00:00, 33.17it/s] 


Encode done: torch.Size([289147, 512]) torch.Size([289147, 512])


In [ ]:
import numpy as np

np.save("/kaggle/working/image_embeddings.npy", image_embeddings.numpy())
np.save("/kaggle/working/text_embeddings.npy", text_embeddings.numpy())

metadata = pd.DataFrame(valid_rows)
metadata.to_csv("/kaggle/working/metadata.csv", index=False)

print("Saved to /kaggle/working/")

Saved all files to /kaggle/working/
